### Dynamic Tools

In [2]:
from dataclasses import dataclass
from typing import Dict, Any, Callable
from langchain_community.utilities import SQLDatabase
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langchain.tools import tool

Globals

In [3]:
model = f"bedrock:us.anthropic.claude-opus-4-6-v1"

Tools

In [4]:
@tool("squared", description="Calculate the squared of a number")
def squared_tool(x: float) -> float:
    return x ** 2


squared_tool.invoke({"x": 467})

218089.0

In [5]:
@tool("square_root", description="Calculate the square root of a number")
def square_root_tool(x: float) -> float:
    return x ** 0.5


square_root_tool.invoke({"x": 467})

21.61018278497431

Data Class

In [6]:
@dataclass
class UserRole:
    user_role: str = "external"

Model Middle Ware

In [7]:
@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [square_root_tool] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

Agent

In [8]:
agent = create_agent(
    model,
    tools=[squared_tool, square_root_tool],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is 12 squared?")]},
    context={"user_role": "external"}
)

for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

What is 12 squared?
================================== Ai Message ==================================

12 squared is **144**. (12 × 12 = 144)

This is a straightforward multiplication, so no tools were needed for this calculation! Let me know if you'd like help with anything else.
